<a href="https://colab.research.google.com/github/firstsignal/activation-geometry-sentiment/blob/main/universal_trajectory_alignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformer-lens


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.7 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=55b3cff2a1bee1fecc3e120c4359c64578555b362596b89a5a7fc38821c89dfb
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator


In [10]:
import torch
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("pythia-70m")

# the matched pair from Chapter 2 — identical except one word
pair_pos = "The meal at the restaurant was absolutely wonderful and the evening felt complete."
pair_neg = "The meal at the restaurant was absolutely terrible and the evening felt complete."

tp = model.to_tokens(pair_pos); tn = model.to_tokens(pair_neg)
_, cp = model.run_with_cache(tp); _, cn = model.run_with_cache(tn)

# matched-pair sets for building the per-layer sentiment axis
pos_matched = [
    "The meal at the restaurant was absolutely wonderful.",
    "Her performance in the final act was brilliant.",
    "The weather on the coast stayed lovely all week.",
    "His speech at the ceremony sounded inspiring.",
    "The ending of the novel felt satisfying.",
]
neg_matched = [
    "The meal at the restaurant was absolutely terrible.",
    "Her performance in the final act was dreadful.",
    "The weather on the coast stayed miserable all week.",
    "His speech at the ceremony sounded tedious.",
    "The ending of the novel felt hollow.",
]

def resid_at(prompts, layer):
    vecs = []
    for p in prompts:
        _, c = model.run_with_cache(model.to_tokens(p))
        vecs.append(c["resid_post", layer][0].mean(dim=0))
    return torch.stack(vecs)

# sanity check the flip position before trusting [8:13]
toks = model.to_str_tokens(pair_pos)
for i, t in enumerate(toks): print(i, repr(t))


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loaded pretrained model pythia-70m into HookedTransformer
0 'The'
1 ' meal'
2 ' at'
3 ' the'
4 ' restaurant'
5 ' was'
6 ' absolutely'
7 ' wonderful'
8 ' and'
9 ' the'
10 ' evening'
11 ' felt'
12 ' complete'
13 '.'
